# Explicit Euler Method

自包含文档，只依赖 PyTorch。Flow Matching 推理解初值问题：

$$\text{Find }x(t)\text{ such that }\frac{dx}{dt}=v_\theta(x,t),\quad x(0)=x_{noise},\quad t\in[0,1].$$

对区间 [tₙ,tₙ₊₁] 积分，再将未知积分内的速度冻结为左端速度：

$$x_{n+1}=x_n+\int_{t_n}^{t_{n+1}}v_\theta(x(\tau),\tau)d\tau\approx x_n+h_n v_\theta(x_n,t_n).$$

因此 Explicit Euler Method 为

$$x_{n+1}=x_n+h_n v_\theta(x_n,t_n).$$

每步 1 NFE；局部误差 O(h²)、全局误差 O(h)。


## 从精确积分式到一阶截断误差

令 $f(x,t)=v_\theta(x,t)$，$h=t_{n+1}-t_n$。精确一步是

$$
x(t_n+h)=x_n+\int_0^h f(x(t_n+\tau),t_n+\tau)d\tau.
$$

沿真解在左端 Taylor 展开，记总导数为

$$
\dot f=f_x f+f_t,
$$

则

$$
f(x(t_n+\tau),t_n+\tau)=f_n+\tau\dot f_n+O(\tau^2).
$$

代入积分得到

$$
x(t_n+h)=x_n+h f_n+\frac{h^2}{2}\dot f_n+O(h^3).
$$

Euler 只保留第一项。因此单步局部截断误差是 $O(h^2)$；在长度固定的区间上约有 $O(1/h)$ 步，累积为全局 $O(h)$。这说明它不需要 Jacobian，但也不能仅因速度场看起来平滑就假设大步长仍准确。


In [ ]:
import torch


class ExplicitEulerMethod:
    """Self-contained first-order solver for dx/dt = velocity(x, t)."""

    def __init__(self, velocity):
        self.velocity = velocity
        self.nfe = 0  # Number of decoder/velocity evaluations in one sample call.

    @torch.no_grad()
    def sample(self, x_init, times):
        """Advance through an increasing 1-D time grid with left-end Euler steps."""
        x = x_init
        self.nfe = 0
        for t_now, t_next in zip(times[:-1], times[1:]):
            step = t_next - t_now
            velocity_now = self.velocity(x, t_now)  # Decoder input is (x_n, t_n).
            x = x + step * velocity_now
            self.nfe += 1
        return x


# A constant velocity integrates exactly under Euler on any grid.
times = torch.linspace(0.0, 1.0, 5)
solver = ExplicitEulerMethod(lambda x, t: torch.full_like(x, 2.0))
output = solver.sample(torch.zeros(2, 2), times)
torch.testing.assert_close(output, torch.full_like(output, 2.0))
assert solver.nfe == 4
print("constant-velocity invariant passed; NFE =", solver.nfe)
